In [1]:
from parameter.aparameter import AParameter
from strategy.strategy_factory import StrategyFactory
from transformer.transformer import Transformer
from backtester.backtester import Backtester
from strategy.strategy import Strategy
from diversifier.diversifier import Diversifier
from datetime import datetime
from database.adatabase import ADatabase
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd

In [2]:
market = ADatabase("market")
market.connect()
tickers = market.retrieve("sp500")["ticker"].values
market.disconnect()

In [3]:
queries = []
for strategy in [x.value for x in Strategy]:
    for diversifier in [x.value for x in Diversifier]:
        for ascending in [True,False]:
            query = {}
            query["strategy"] = strategy
            query["diversifier"] = diversifier
            query["stop_loss"] = 1
            query["tickers"] = tickers
            query["ascending"] = ascending
            queries.append(query)

In [4]:
analysis = []
for query in tqdm(queries):
    try:
        param = AParameter()
        param.build(query)
        strat = StrategyFactory.build(param)
        sim = Transformer.transform_v2(strat,datetime.now(),datetime.now())
        trades = Backtester.backtest_v2(strat,sim)
        portfolios = Backtester.portfolio(trades)
        kpi = Backtester.kpi(trades,portfolios)
        results = kpi | strat.__dict__
        results.pop("diversifier_set")
        results.pop("tickers")
        results.pop("prices")
        results.pop("factors")
        analysis.append(results)
    except Exception as e:
        print(str(e))

 79%|████████████████████████████████████████████████████████████████████████████████████▏                      | 85/108 [1:51:52<32:07, 83.82s/it]

single positional indexer is out-of-bounds


 80%|█████████████████████████████████████████████████████████████████████████████████████▏                     | 86/108 [1:52:46<27:26, 74.85s/it]

single positional indexer is out-of-bounds


 81%|██████████████████████████████████████████████████████████████████████████████████████▏                    | 87/108 [1:53:39<23:59, 68.53s/it]

...

 82%|████████████████████████████████████████████████████████████████████████████████████████▏                  | 89/108 [1:55:27<19:19, 61.02s/it]

single positional indexer is out-of-bounds


 83%|█████████████████████████████████████████████████████████████████████████████████████████▏                 | 90/108 [1:56:21<17:38, 58.80s/it]

single positional indexer is out-of-bounds


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 108/108 [2:20:09<00:00, 77.86s/it]


In [5]:
a = pd.DataFrame(analysis)

In [8]:
a.sort_values("return",ascending=False)

,number_of_trades,standard_deviation,coefficient_of_variance,sharpe,return,strategy,diversifier,positions,holding_period,stop_loss,ascending,factors
94,1008,15655.1946,1.6308,4.2900,67160.5120,AVERAGE_RETURN,SIMPLE,1,5,1.0,True,NaN
11,1008,499.8647,1.8437,4.0091,2004.0315,BOLLINGER_WIDTH,SIMPLE,1,5,1.0,False,NaN
9,3024,304.3602,1.8348,3.7117,1129.6952,BOLLINGER_WIDTH,INDEX_CORRELATION,3,5,1.0,False,NaN
100,1008,255.7487,1.3670,3.0740,786.1666,MOVING_AVERAGE,SIMPLE,1,5,1.0,True,NaN
83,988,192.0012,1.2680,3.9704,762.3241,UNITY_AI,SIMPLE,1,5,1.0,False,"[stochastic_oscillator, macd, bollinger, coeff..."
...,...,...,...,...,...,...,...,...,...,...,...,...
51,3024,0.2481,0.1909,7.1051,1.7631,STANDARD_DEVIATION,INDEX_CORRELATION,3,5,1.0,False,NaN
2,3024,0.2532,0.1998,4.7001,1.1901,BOLLINGER,INDEX_CORRELATION,3,5,1.0,True,NaN
46,1008,0.4740,0.4043,2.4416,1.1574,STOCHASTIC_OSCILLATOR,SIMPLE,1,5,1.0,True,NaN
101,1008,0.3162,0.4075,2.6380,0.8341,MOVING_AVERAGE,SIMPLE,1,5,1.0,False,NaN
